# Kiểm thử worker HTTP và Langfuse theo trace_id

AWS dùng Moto; HTTP processing và Langfuse dùng MockTransport, không gọi dịch vụ thật. Kiểm tra giao thức multipart, polling và lưu output trước ACK SQS. Cài `pip install -e ".[test]"`, chọn kernel project.


In [ ]:
from pathlib import Path
import sys, unittest
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workers/main.py").is_file())
for path in (root, root / "tests/unit"):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
from test_workers import WorkerAWSTests, WorkerBoundaryTests


## SQS → S3 → HTTP processing → Langfuse → S3/DynamoDB → ACK

Kiểm tra runtime tự sinh, trace/idempotency ổn định, multipart bytes và chỉ xóa SQS sau lưu output Agent 3. Các service đích được giả lập.


In [ ]:
suite = unittest.TestSuite([WorkerAWSTests("test_http_dispatch_trace_poll_and_persist_before_sqs_delete")])
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


## Retry, lease, timeout, remote failure và lỗi ghi AWS

Kiểm tra dùng lại runtime/trace/Idempotency-Key, lease, event sai thứ tự, input thay đổi và lỗi S3/DynamoDB. Không đánh dấu completed khi trace timeout hoặc stage thất bại.


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(WorkerAWSTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


## Tiến trình ghi riêng và ACK qua IPC

Spawn một tiến trình writer thực; Moto được khởi tạo trong tiến trình con. Kiểm tra timeout/ACK lỗi và từ chối event cũ sau hoàn tất. Đây chưa phải kiểm chứng IAM, Docker hay AWS thật.


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(WorkerBoundaryTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


## Giao thức API và giới hạn connector

Kiểm tra cursor (kể cả trang rỗng), Basic auth, JSON output, identity, sequence, lỗi HTTP và giới hạn response.


In [ ]:
from test_worker_connectors import ConnectorTests
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ConnectorTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
